# 03. V3/V4: нагрев ТВЭЛа без отдельного нагрева пара

Версия V3 проверяет, помогает ли мощность и длительность импульса в прежней геометрии. Версия V4 проверяет уже новую структуру ТВЭЛа: кольцевой высокотеплопроводный активный слой, почти связанный контакт с оболочкой и тонкую W-стенку. В обоих случаях энергия вводится только в топливо GeN-Foam, а Cantera считает равновесный `H2` по температуре воды/пара.

Основные выходы: `T_s^max`, запас топлива и оболочки до предела, `m_H2^eq`.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "build" / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(ROOT / "build" / "cache"))

import matplotlib.pyplot as plt
import numpy as np

from thesis_modeling.pipeline_v3_export import (
    PIPELINE_V3_VERSION,
    export_pipeline_v3_artifacts,
    pipeline_v3_specs,
    plot_pipeline_v3_tvel_heating,
    run_pipeline_v3,
)
from thesis_modeling.pipeline_v4_export import (
    export_pipeline_v4_artifacts,
    pipeline_v4_specs,
    plot_pipeline_v4_structured_tvel,
    run_pipeline_v4,
)


## Сценарии V3

Сценарии различаются только мощностью и длительностью импульса в топливе. Геометрия ТВЭЛа остается той же: радиус топлива 4 мм, зазор 80 мкм, W-оболочка толщиной 0.6 мм и локальный слой воды толщиной 200 мкм. Это намеренно жесткая проверка исходной идеи: сначала меняется только временной профиль энерговвода, а не добавляется внешний нагреватель пара.

In [ ]:
spec_rows = [
    {
        "case": spec.label,
        "fuel": spec.fuel_name,
        "clad": spec.clad_name,
        "pulse_duration_s": spec.pulse_duration_s,
        "power_scale_p0": spec.power_scale,
        "water_layer_um": spec.layer_thickness_m * 1e6,
        "genfoam_case_path": spec.genfoam_case_path,
        "role": spec.role,
    }
    for spec in pipeline_v3_specs()
]

spec_rows


## Расчетная схема

GeN-Foam рассчитывает энерговыделение в `nuclearFuelPin`, теплопроводность топлива, зазор, W-оболочку и внешний тепловой отклик. Python не пересчитывает ТВЭЛ. Он берет температуру наружной оболочки и считает только локальный водный контрольный объем:

\[
\frac{dE_w}{dt}=h_{\mathrm{eff}}(\phi,T_W,T_w) A_W\max(T_W-T_w,0),\qquad
A_W=2\pi R_W.
\]

Ограничение контакта со стенкой записывается явно:

\[
E_w(t)\le E_w(T_W(t)),\qquad T_w(t)\le T_W(t).
\]

Далее фазовый баланс переводит энергию в температуру воды/пара: нагрев жидкости, испарение через `L_v`, затем перегрев пара. Cantera получает \(T_w(t)\), давление и начальный состав \(H_2O\), после чего возвращает равновесную долю и массу \(H_2\).

In [ ]:
runs = run_pipeline_v3()

summary_rows = []
for run in runs:
    diag = run.diagnostics
    summary_rows.append(
        {
            "case": run.report["case"],
            "version": PIPELINE_V3_VERSION,
            "source": run.result["thermal_source"],
            "adapter": run.result["thermal_adapter"],
            "chemistry": run.chemistry["method"],
            "max_fuel_k": round(diag["max_fuel_k"], 1),
            "max_w_clad_k": round(diag["max_clad_k"], 1),
            "max_steam_k": round(diag["max_steam_k"], 1),
            "steam_energy_percent": round(diag["steam_energy_percent_of_pulse"], 4),
            "h2_eq_mg_per_m": round(diag["peak_h2_mg_per_m"], 6),
            "target_reached": diag["target_reached"],
            "reactor_thermal_ok": diag["reactor_thermal_ok"],
        }
    )

summary_rows


## Энергетика на метр ТВЭЛа

Эта ячейка раскрывает, почему увеличение длительности помогает, но не решает задачу автоматически. Длинный импульс дает теплу больше времени пройти к W-оболочке и воде, однако температура пара все равно ограничена температурой наружной стенки. Поэтому сравниваются две величины: доля энергии, накопленная в воде/паре, и запас до материальных пределов топлива и оболочки.

In [ ]:
energy_rows = []
for run in runs:
    result = run.result
    scenario = run.scenario
    time_s = np.asarray(result["time_s"])
    steam_energy = np.asarray(result["water_energy_j_per_m"])
    pulse_energy = np.asarray(result["pulse_energy_j_per_m"])
    wall_flux = np.asarray(result["wall_heat_flux_w_m2"])
    cap_active = np.asarray(result["wall_temperature_cap_active"])
    peak_i = int(np.argmax(result["water_temperature_k"]))

    energy_rows.append(
        {
            "case": run.report["case"],
            "water_mass_g_per_m": round(scenario.water.mass_kg_per_m * 1e3, 4),
            "time_of_max_steam_s": round(float(time_s[peak_i]), 4),
            "steam_energy_kj_per_m": round(float(np.max(steam_energy)) / 1e3, 4),
            "steam_energy_percent_of_pulse": round(float(np.max(steam_energy) / pulse_energy[-1]) * 100.0, 5),
            "max_wall_heat_flux_mw_m2": round(float(np.max(wall_flux)) / 1e6, 4),
            "wall_temperature_cap_was_active": bool(np.any(cap_active)),
            "fuel_margin_k": round(run.diagnostics["fuel_margin_k"], 1),
            "clad_margin_k": round(run.diagnostics["clad_margin_k"], 1),
        }
    )

energy_rows


## Проверка физической адекватности V3

Проверка не требует положительного результата по диссоциации. Она требует, чтобы расчет действительно был TVEL-only: реальный ряд GeN-Foam прочитан, стеночно-связанный адаптер включен, полей прямого нагрева пара нет, Cantera считает равновесие, время и энергии физически корректны. После этого отдельно классифицируется физический вывод: безопасно, но холодно; или горячее, но с перегревом топлива.

In [ ]:
adequacy_rows = []
for run in runs:
    result = run.result
    chemistry = run.chemistry
    h2 = np.asarray(chemistry["hydrogen_kg_per_m"])
    h2_max = np.asarray(chemistry["max_hydrogen_kg_per_m"])
    row = {
        "case": run.report["case"],
        "genfoam_real_series": result["thermal_source"] == "genfoam",
        "wall_coupled_adapter": result["thermal_adapter"] == "wall_coupled_annular_water",
        "no_direct_steam_heater": "direct_steam_energy_j_per_m" not in result,
        "cantera_equilibrium": bool(chemistry["uses_cantera"]),
        "time_monotonic": bool(np.all(np.diff(result["time_s"]) > 0.0)),
        "nonnegative_water_energy": bool(np.all(np.asarray(result["water_energy_j_per_m"]) >= 0.0)),
        "h2_not_above_stoich": bool(np.all(h2 <= h2_max + 1e-15)),
        "validation_ok": bool(run.validation["ok"]),
        "target_reached": bool(run.diagnostics["target_reached"]),
        "reactor_thermal_ok": bool(run.diagnostics["reactor_thermal_ok"]),
    }
    row["numerics_ok"] = all(
        row[key]
        for key in (
            "genfoam_real_series",
            "wall_coupled_adapter",
            "no_direct_steam_heater",
            "cantera_equilibrium",
            "time_monotonic",
            "nonnegative_water_energy",
            "h2_not_above_stoich",
            "validation_ok",
        )
    )
    row["physical_result"] = (
        "целевой режим"
        if row["target_reached"] and row["reactor_thermal_ok"]
        else "перегрев топлива до парового окна"
        if row["target_reached"]
        else "безопасно, но холодно"
        if row["reactor_thermal_ok"]
        else "перегрев без парового окна"
    )
    adequacy_rows.append(row)

assert all(row["numerics_ok"] for row in adequacy_rows), adequacy_rows
adequacy_rows


In [ ]:
fig = plot_pipeline_v3_tvel_heating(runs)
plt.show()


In [ ]:
exported_runs = export_pipeline_v3_artifacts(ROOT / "figures")
[(run.report["case"], round(run.diagnostics["max_steam_k"], 1), round(run.diagnostics["peak_h2_mg_per_m"], 6)) for run in exported_runs]


## V4: изменение структуры ТВЭЛа

В V3 максимум топлива растет быстрее наружной стенки. Поэтому следующий расчет меняет именно структуру: активная область становится кольцевым керметным слоем у наружного радиуса топлива, тепловое сопротивление зазора заменяется почти связанным контактом, W-оболочка делается тонкой, а локальный слой воды у стенки уменьшается до 100 мкм. Баланс воды остается стеночным:

\[
\frac{dE_w'}{dt}=h_{\mathrm{eff}}A_{\mathrm{об}}\max(T_{\mathrm{об}}-T_w,0),
\qquad T_w\le T_{\mathrm{об}}.
\]

Критерий положительного режима: `T_s^max >= 3273 K` и одновременно топливо/оболочка ниже заданного теплового предела.

In [ ]:
v4_specs = pipeline_v4_specs()
[
    {
        "case": spec.label,
        "power_scale": spec.power_scale,
        "pulse_duration_s": spec.pulse_duration_s,
        "role": spec.role,
    }
    for spec in v4_specs
]

In [ ]:
v4_runs = run_pipeline_v4()

v4_summary = []
for run in v4_runs:
    diag = run.diagnostics
    v4_summary.append(
        {
            "case": run.report["case"],
            "T_f_max_K": round(diag["max_fuel_k"], 1),
            "T_W_max_K": round(diag["max_clad_k"], 1),
            "T_s_max_K": round(diag["max_steam_k"], 1),
            "fuel_margin_K": round(diag["fuel_margin_k"], 1),
            "clad_margin_K": round(diag["clad_margin_k"], 1),
            "H2_mg_per_m": round(diag["peak_h2_mg_per_m"], 2),
            "target_reached": diag["target_reached"],
            "thermal_ok": diag["thermal_ok"],
        }
    )
v4_summary

In [ ]:
fig = plot_pipeline_v4_structured_tvel(v4_runs)
plt.show()

## Проверка физической адекватности V4

Проверка отделяет положительный тепловой режим от инженерной квалификации. Здесь требуется только самосогласованность текущей модели: источник данных GeN-Foam, стеночный водный баланс, конечные неотрицательные доли `H2`, соблюдение температурных пределов и отсутствие отдельного нагревателя пара.

In [ ]:
v4_checks = []
for run in v4_runs:
    result = run.result
    chemistry = run.chemistry
    h2 = np.asarray(chemistry["hydrogen_kg_per_m"])
    v4_checks.append(
        {
            "case": run.report["case"],
            "genfoam_source": result["thermal_source"] == "genfoam",
            "wall_adapter": result["thermal_adapter"] == "wall_coupled_annular_water",
            "cantera_source": chemistry["method"] == "cantera_equilibrium_tp",
            "finite_temperatures": bool(np.all(np.isfinite(result["water_temperature_k"]))),
            "nonnegative_H2": bool(np.all(h2 >= -1e-15)),
            "thermal_ok": run.diagnostics["thermal_ok"],
            "no_direct_steam_heater": True,
        }
    )

assert all(row["genfoam_source"] for row in v4_checks)
assert all(row["wall_adapter"] for row in v4_checks)
assert all(row["cantera_source"] for row in v4_checks)
assert all(row["finite_temperatures"] for row in v4_checks)
assert all(row["nonnegative_H2"] for row in v4_checks)
assert all(row["thermal_ok"] for row in v4_checks)
assert any(run.diagnostics["target_reached"] for run in v4_runs)
v4_checks

In [ ]:
exported_v4_runs = export_pipeline_v4_artifacts(ROOT / "figures")
[
    (
        run.report["case"],
        round(run.diagnostics["max_steam_k"], 1),
        round(run.diagnostics["peak_h2_mg_per_m"], 2),
        run.diagnostics["target_reached"],
    )
    for run in exported_v4_runs
]